<a href="https://colab.research.google.com/github/Nboev2004/InClassAssignments/blob/main/HW7_Strings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Python Homework - Cleaning Data & Feature Engineering

## Packages Needed

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# These for the Linear Regression models in Part 2
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Part 1 - Data Cleaning + Basic Reporting

## Request from management

A regional restaurant chain wants to compare sales by **region, store type, and sales channel**. The raw extract contains one row for each **month x store** (plus a deliberate duplicate). Dine-in, delivery, and drive-thru sales are stored as separate columns.

### 1.1 Load and inspect

Load the file and show the first few rows, shape, and data types. Check for missing data and exact duplicate rows.

**QUESTION: What "problems" do you see in this data? What will need to be fixed?**

+ *answer 1*
+ *answer 2*
+ *add more if needed*

The dataset contains a deliberate duplicate that needs to be removed to avoid counting sales twice.



The month variable needs to be converted into a proper date format.



The three sales channel columns contain currency values that need to be converted into numeric values.



Region and store type information needs to be separated into individual columns and standardized.

In [2]:
sales = pd.read_csv('https://dxl-datasets.s3.us-east-1.amazonaws.com/data/hw/restaurant_sales.csv')

In [27]:
# YOUR CODE HERE

# Display first five rows
display(sales.head())

# Display dataset dimensions
print("Shape:", sales.shape)

# Display data types
print("\nData Types:")
print(sales.dtypes)

# Check missing values
print("\nMissing Values:")
print(sales.isna().sum())

# Check exact duplicates
print("\nDuplicate Rows:", sales.duplicated().sum())


,month,store_id,region_store_type,Dine-In,Delivery,Drive-Thru
0,2026-04-01,R4U2,West - Urban,"$40,237.23","$26,647.14","$26,300.23"
1,2026-05-01,R4S1,West - suburban,"$43,510.87","$30,506.42","$30,815.93"
2,2026-06-01,R4U2,western - Urban,"$52,825.73","$27,303.62","$41,615.95"
3,2026-05-01,R3U2,mid west - Urban,"$31,229.35","$18,320.99","$27,979.30"
4,2026-12-01,R3U1,MW - Urban,"$25,925.22","$15,437.39","$24,554.85"


Shape: (193, 6)

Data Types:
month                object
store_id             object
region_store_type    object
Dine-In              object
Delivery             object
Drive-Thru           object
dtype: object

Missing Values:
month                0
store_id             0
region_store_type    0
Dine-In              0
Delivery             0
Drive-Thru           0
dtype: int64

Duplicate Rows: 1


### 1.2 Remove duplicate records

Remove duplicates and confirm none remain.

In [28]:
# YOUR CODE HERE

sales = sales.drop_duplicates().copy()

print("Duplicate rows remaining:", sales.duplicated().sum())
print("Updated dataset shape:", sales.shape)


Duplicate rows remaining: 0
Updated dataset shape: (192, 6)


### 1.3 Convert the date variable

**QUESTION: What data type was `month` before converting it? What data type is it after converting it?**

+ *answer here*



The month variable was originally stored as an object (string) data type. After converting it using pd.to_datetime(), it becomes datetime64[ns]. This allows Python to recognize the values as dates and makes it easier to sort and analyze sales over time.

In [29]:
# YOUR CODE HERE

print("Before conversion:", sales["month"].dtype)

sales["month"] = pd.to_datetime(sales["month"])

print("After conversion:", sales["month"].dtype)

display(sales.head())


Before conversion: object
After conversion: datetime64[ns]


,month,store_id,region_store_type,Dine-In,Delivery,Drive-Thru
0,2026-04-01,R4U2,West - Urban,"$40,237.23","$26,647.14","$26,300.23"
1,2026-05-01,R4S1,West - suburban,"$43,510.87","$30,506.42","$30,815.93"
2,2026-06-01,R4U2,western - Urban,"$52,825.73","$27,303.62","$41,615.95"
3,2026-05-01,R3U2,mid west - Urban,"$31,229.35","$18,320.99","$27,979.30"
4,2026-12-01,R3U1,MW - Urban,"$25,925.22","$15,437.39","$24,554.85"


### 1.4 Convert currency variables

**SOMETHING TO THINK ABOUT:** Here the sales channel is pivoted across separate columns. For this reason, you need to convert three separate currency columns. Had this data been in long form - which I'll ask you to conver to later - you would just have a single revenue column to fix. With 3 variables this might not seem bad, but if I had many it would be shorter to `melt`, convert, then `pivot` back rather than separate convert each of several columns.

In [30]:
# Your Code Here

sales_columns = ["Dine-In", "Delivery", "Drive-Thru"]

for col in sales_columns:
    sales[col] = pd.to_numeric(
        sales[col].astype(str)
        .str.replace(r"[$,]", "", regex=True)
        .str.strip(),
        errors="coerce"
    )

print(sales[sales_columns].dtypes)

display(sales.head())


Dine-In       float64
Delivery      float64
Drive-Thru    float64
dtype: object


,month,store_id,region_store_type,Dine-In,Delivery,Drive-Thru
0,2026-04-01,R4U2,West - Urban,40237.23,26647.14,26300.23
1,2026-05-01,R4S1,West - suburban,43510.87,30506.42,30815.93
2,2026-06-01,R4U2,western - Urban,52825.73,27303.62,41615.95
3,2026-05-01,R3U2,mid west - Urban,31229.35,18320.99,27979.30
4,2026-12-01,R3U1,MW - Urban,25925.22,15437.39,24554.85


### 1.5 Separate Region and Store Type

**QUESTION: Why even both splitting them into their own columns?**

+ *answer here*


Splitting region and store type into separate columns makes the data easier to organize and analyze. It allows us to compare sales across different regions and store types independently instead of having both pieces of information combined in one column.

In [36]:
# Your Code Here

# Split region_store_type into two separate columns
parts = sales["region_store_type"].str.split("-", n=1, expand=True)

sales["region"] = parts[0].str.strip()
sales["store_type"] = parts[1].str.strip()

# Standardize region names
sales["region"] = (
    sales["region"]
    .str.lower()
    .str.strip()
    .replace({
        "western": "west",
        "mid west": "midwest",
        "mw": "midwest"
    })
    .str.title()
)

# Display the results
display(sales[["region_store_type", "region", "store_type"]].head(10))



,region_store_type,region,store_type
0,West - Urban,West,Urban
1,West - suburban,West,suburban
2,western - Urban,West,Urban
3,mid west - Urban,Midwest,Urban
4,MW - Urban,Midwest,Urban
5,NE - Urban,Ne,Urban
6,W - suburban,W,suburban
7,Northeast - Urban,Northeast,Urban
8,Southeast - Urban,Southeast,Urban
9,NE - Urban,Ne,Urban


### 1.6 Standard Store Type

Standardize the store type column you just created by standardizing the case.

In [37]:
# Your Code Here

sales["store_type"] = sales["store_type"].str.strip().str.title()
print(sales["store_type"].value_counts())


store_type
Urban       96
Suburban    96
Name: count, dtype: int64


### 1.7 Standardize Region

Standardize region using `.str.contains()` to find and group similar labels into `Northeast`, `Southeast`, `Midwest`, and `West`.

**HINT:** You may run into an issue using `.str.contains()` with simple search patterns because west and midwest overlap. Try searching:
+ `'^w'` or `'^m'`

to find words that start with w or start with m, rather than searching "west" or "midwest". Be sure to set case to false.

In [38]:
# Your Code Here

# Standardize Northeast
sales.loc[
    sales["region"].str.contains(r"^n", case=False, na=False),
    "region"
] = "Northeast"

# Standardize Southeast
sales.loc[
    sales["region"].str.contains(r"^s", case=False, na=False),
    "region"
] = "Southeast"

# Standardize Midwest
sales.loc[
    sales["region"].str.contains(r"^m", case=False, na=False),
    "region"
] = "Midwest"

# Standardize West
sales.loc[
    sales["region"].str.contains(r"^w", case=False, na=False),
    "region"
] = "West"

# Display standardized regions
print(sales["region"].value_counts())


region
West         48
Midwest      48
Northeast    48
Southeast    48
Name: count, dtype: int64


### 1.8 Sales by Region

In each of the cells below, report the requested table.

In [39]:
# a) Total Sales by Region

# Calculate total sales across all channels
sales["total_sales"] = sales[
    ["Dine-In", "Delivery", "Drive-Thru"]
].sum(axis=1)

# Group total sales by region
region_sales = sales.groupby("region")["total_sales"].sum()

display(region_sales)


,total_sales
region,
Midwest,4116026.71
Northeast,5123835.15
Southeast,4866883.43
West,4802537.44


In [40]:
# b) Sales for each Channel by Region

# Calculate sales for each channel by region
channel_sales = sales.groupby("region")[
    ["Dine-In", "Delivery", "Drive-Thru"]
].sum()

display(channel_sales)


,Dine-In,Delivery,Drive-Thru
region,,,
Midwest,1755073.70,939175.74,1421777.27
Northeast,2154902.33,1227078.87,1741853.95
Southeast,2037309.41,1188146.50,1641427.52
West,2029621.54,1156919.10,1615996.80


### 1.9 Reshape into "long" form

Use `.melt()` to turn `Dine-In`, `Delivery`, and `Drive-Thru` into a single `sales_channel` column and a `revenue` column. Preserve `month`, `store_id`, `region`, and `store_type`.

**QUESTION: What does one row of this dataset represent after reshaping?**

+ *answer here*

Each row represents the revenue generated by one specific sales channel at a particular store during a specific month. The three separate sales columns are combined into one sales_channel column and one revenue column.

In [41]:
# Your Code Here

# Reshape sales data from wide format to long format
sales_long = sales.melt(
    id_vars=["month", "store_id", "region", "store_type"],
    value_vars=["Dine-In", "Delivery", "Drive-Thru"],
    var_name="sales_channel",
    value_name="revenue"
)

# Display the reshaped data
display(sales_long.head(10))

print("Original dataset shape:", sales.shape)
print("Long format dataset shape:", sales_long.shape)


,month,store_id,region,store_type,sales_channel,revenue
0,2026-04-01,R4U2,West,Urban,Dine-In,40237.23
1,2026-05-01,R4S1,West,Suburban,Dine-In,43510.87
2,2026-06-01,R4U2,West,Urban,Dine-In,52825.73
3,2026-05-01,R3U2,Midwest,Urban,Dine-In,31229.35
4,2026-12-01,R3U1,Midwest,Urban,Dine-In,25925.22
5,2026-10-01,R1U1,Northeast,Urban,Dine-In,34156.14
6,2026-02-01,R4S2,West,Suburban,Dine-In,38009.63
7,2026-07-01,R1U2,Northeast,Urban,Dine-In,57038.47
8,2026-06-01,R2U2,Southeast,Urban,Dine-In,55427.69
9,2026-08-01,R1U2,Northeast,Urban,Dine-In,52571.90


Original dataset shape: (192, 9)
Long format dataset shape: (576, 6)


### 1.10 Sale by Region

In each cell below, report the requested table.

**QUESTION: What was easier, calculating these tables with the data in wide form (1.8) or in wide form (here), and why?**

+ *answer here*

I found the long format easier for comparing sales across different channels because all the revenue values are stored in one column. Using groupby() makes it simple to organize sales by region and sales channel without having to work with multiple separate columns. However, the wide format was also convenient for calculating total sales by region.

In [42]:
# a) Total Sales by Region

# Calculate total revenue by region using long format
long_region_sales = sales_long.groupby("region")["revenue"].sum()

display(long_region_sales)


,revenue
region,
Midwest,4116026.71
Northeast,5123835.15
Southeast,4866883.43
West,4802537.44


In [43]:
# b) Sales for each Channel by Region

# Calculate revenue by region and sales channel
long_channel_sales = sales_long.groupby(
    ["region", "sales_channel"]
)["revenue"].sum().unstack()

display(long_channel_sales)


sales_channel,Delivery,Dine-In,Drive-Thru
region,,,
Midwest,939175.74,1755073.70,1421777.27
Northeast,1227078.87,2154902.33,1741853.95
Southeast,1188146.50,2037309.41,1641427.52
West,1156919.10,2029621.54,1615996.80


# Part 2 - Feature Engineering from Used-Car Descriptions

## Scenario

A used-car dealership predicts selling prices from a few structured characteristics: **vehicle age, mileage, and whether it is a premium brand**. Descriptions may contain important details that are currently being overlooked.

Use `.str.contains()` to create binary features from listing descriptions, then compare a supplied baseline regression with an expanded model. You do **not** need to understand the mechanics of linear regression or edit the model code.


### 2.1 Load and inspect

Look at the listing description.

**QUESTION: What information about a car is buried in the text that might be useful for predicting its sale price?**

+ *answer here*

The listing descriptions contain additional information that could be useful for predicting a car's selling price. For example, they may mention whether the vehicle had one previous owner, any accident or collision history, recent maintenance or repairs, and whether it includes a premium package. These details can affect a vehicle's condition and value, even when two cars have similar mileage and age.

In [15]:
cars = pd.read_csv("https://dxl-datasets.s3.us-east-1.amazonaws.com/data/hw/used_cars.csv")
pd.set_option("display.max_colwidth", 180)
display(cars.head(8))

,listing_id,make,model,year,mileage,listing_description,sale_price
0,CAR0001,Chevrolet,Malibu,2023,38233,2023 Chevrolet Malibu. Financing options available. Single-owner car.,31419
1,CAR0002,Kia,K5,2015,175115,2015 Kia K5. Contact our team for details. Minor accident reported.,5358
2,CAR0003,Honda,Civic,2021,42945,2021 Honda Civic. Clean interior.,26543
3,CAR0004,BMW,3 Series,2015,121388,2015 BMW 3 Series. Runs and drives well. Fresh oil change and service.,22110
4,CAR0005,Ford,Focus,2014,130728,2014 Ford Focus. Financing options available. One owner vehicle. Accident history disclosed.,4500
5,CAR0006,Ford,Escape,2013,154166,2013 Ford Escape. Runs and drives well. Only one previous owner. Premium package included.,13281
6,CAR0007,Honda,CR-V,2018,60218,2018 Honda CR-V. Available for test drive.,25154
7,CAR0008,Mercedes-Benz,GLC,2020,53905,2020 Mercedes-Benz GLC. Contact our team for details. Prior collision documented.,32029


In [44]:
# Your Code Here

# Check dataset dimensions
print("Dataset shape:", cars.shape)

# Check column data types
print("\nData Types:")
print(cars.dtypes)

# Check for missing values
print("\nMissing Values:")
print(cars.isna().sum())

# Display listing descriptions
print("\nSample Listing Descriptions:")
display(cars["listing_description"].head(10))


Dataset shape: (1200, 13)

Data Types:
listing_id             object
make                   object
model                  object
year                    int64
mileage                 int64
listing_description    object
sale_price              int64
vehicle_age             int64
premium_brand           int64
one_owner               int64
accident_history        int64
recent_service          int64
premium_package         int64
dtype: object

Missing Values:
listing_id             0
make                   0
model                  0
year                   0
mileage                0
listing_description    0
sale_price             0
vehicle_age            0
premium_brand          0
one_owner              0
accident_history       0
recent_service         0
premium_package        0
dtype: int64

Sample Listing Descriptions:


,listing_description
0,2023 Chevrolet Malibu. Financing options available. Single-owner car.
1,2015 Kia K5. Contact our team for details. Minor accident reported.
2,2021 Honda Civic. Clean interior.
3,2015 BMW 3 Series. Runs and drives well. Fresh oil change and service.
4,2014 Ford Focus. Financing options available. One owner vehicle. Accident history disclosed.
5,2013 Ford Escape. Runs and drives well. Only one previous owner. Premium package included.
6,2018 Honda CR-V. Available for test drive.
7,2020 Mercedes-Benz GLC. Contact our team for details. Prior collision documented.
8,2022 Toyota Corolla. Financing options available. One owner vehicle. Upgraded luxury trim.
9,2013 Chevrolet Equinox. Clean interior. Accident history disclosed. Premium options package.


## 2.2 Create structured predictors

Create `vehicle_age` and `premium_brand` (1 for BMW or Mercedes-Benz, 0 otherwise).

In [20]:
# YOUR CODE HERE

# Identify the correct column names
print("Available columns:", cars.columns.tolist())

# Identify year and brand columns
year_col = next(
    (c for c in ["year", "model_year"] if c in cars.columns),
    None
)

brand_col = next(
    (c for c in ["brand", "make"] if c in cars.columns),
    None
)

# Create vehicle age
cars["vehicle_age"] = pd.Timestamp.today().year - pd.to_numeric(
    cars[year_col], errors="coerce"
)

# Create premium brand indicator
cars["premium_brand"] = (
    cars[brand_col]
    .astype(str)
    .str.contains(r"BMW|Mercedes-Benz", case=False, na=False)
    .astype(int)
)

# Verify the new columns
display(cars[["vehicle_age", "mileage", "premium_brand"]].head())


Available columns: ['listing_id', 'make', 'model', 'year', 'mileage', 'listing_description', 'sale_price']


,vehicle_age,mileage,premium_brand
0,3,38233,0
1,11,175115,0
2,5,42945,0
3,11,121388,1
4,12,130728,0


## 2.3 Baseline Model

Execute this cell after creating the two variables above. Record RMSE and R². **Lower RMSE and higher R² are better.**

The baseline uses vehicle age, mileage, and premium-brand status. The same train/test split will be used for both models.

In [21]:
# DO NOT CHANGE THIS BLOCK, JUST RUN IT

baseline_features = ["vehicle_age", "mileage", "premium_brand"]
y = cars["sale_price"]
train_idx, test_idx = train_test_split(np.arange(len(cars)), test_size=0.25, random_state=123)

baseline_model = LinearRegression()
baseline_model.fit(cars.iloc[train_idx][baseline_features], y.iloc[train_idx])
baseline_predictions = baseline_model.predict(cars.iloc[test_idx][baseline_features])
baseline_rmse = np.sqrt(mean_squared_error(y.iloc[test_idx], baseline_predictions))
baseline_r2 = r2_score(y.iloc[test_idx], baseline_predictions)
print(f"Baseline RMSE: ${baseline_rmse:,.0f}")
print(f"Baseline R²: {baseline_r2:.3f}")

Baseline RMSE: $3,408
Baseline R²: 0.894


## 2.4 Engineer Features from Description

Create 0/1 variables using `.str.contains(..., case=False).astype(int)`:

- `one_owner` — one owner, single-owner, one previous owner
- `accident_history` - ??
- `recent_service` - ??
- `premium_package` - ??

Here is a worked example for `one_owner`. Create the **other three yourself**. Consider synonyms, not just one exact word, as each feature appears in the descriptions in different way.

**QUESTION: How many cars were flagged for each feature?**

+ *answer here*

The number of cars flagged for each feature was:
- One Owner: 516 cars
- Accident History: 267 cars
- Recent Service: 331 cars
- Premium Package: 98 cars
One-owner status was the most common feature, followed by recent service and accident history. Premium packages were the least common feature among the four categories.

In [45]:
print("One Owner:", cars["one_owner"].sum())
print("Accident History:", cars["accident_history"].sum())
print("Recent Service:", cars["recent_service"].sum())
print("Premium Package:", cars["premium_package"].sum())

One Owner: 516
Accident History: 267
Recent Service: 331
Premium Package: 98


In [22]:
cars["one_owner"] = (
    cars["listing_description"]
    .str.contains("one owner|single-owner|single owner|one previous owner", case=False)
    .astype(int)
)

cars["one_owner"].value_counts()

,count
one_owner,
0,684
1,516


In [25]:
# Your Code Here

# Get the vehicle listing descriptions
description = cars["listing_description"].fillna("")

# Identify accident history
cars["accident_history"] = description.str.contains(
    r"accident|collision|damage history",
    case=False,
    regex=True
).astype(int)

# Identify recent service
cars["recent_service"] = description.str.contains(
    r"recent service|recently serviced|maintenance|new tires|new brakes|"
    r"service records|fresh oil change",
    case=False,
    regex=True
).astype(int)

# Identify premium packages
cars["premium_package"] = description.str.contains(
    r"premium package|luxury package|technology package|sport package|"
    r"executive package|premium trim|navigation package",
    case=False,
    regex=True
).astype(int)

# Count how many cars were flagged for each feature
print("One Owner:")
print(cars["one_owner"].sum())

print("\nAccident History:")
print(cars["accident_history"].sum())

print("\nRecent Service:")
print(cars["recent_service"].sum())

print("\nPremium Package:")
print(cars["premium_package"].sum())


One Owner:
516

Accident History:
267

Recent Service:
331

Premium Package:
98


## 2.5 Expanded Model
The new model adds your four text features to the original variables. Run the supplied code and compare RMSE and R² with the baseline.

In [26]:
# DO NOT MODIFY THIS CODE, JUST RUN IT AFTER CREATING THE 4 FEATURES ABOVE
text_features = ["one_owner", "accident_history", "recent_service", "premium_package"]
expanded_features = baseline_features + text_features
expanded_model = LinearRegression()
expanded_model.fit(cars.iloc[train_idx][expanded_features], y.iloc[train_idx])
expanded_predictions = expanded_model.predict(cars.iloc[test_idx][expanded_features])
expanded_rmse = np.sqrt(mean_squared_error(y.iloc[test_idx], expanded_predictions))
expanded_r2 = r2_score(y.iloc[test_idx], expanded_predictions)
print(f"Baseline: RMSE ${baseline_rmse:,.0f} | R² {baseline_r2:.3f}")
print(f"Expanded: RMSE ${expanded_rmse:,.0f} | R² {expanded_r2:.3f}")

Baseline: RMSE $3,408 | R² 0.894
Expanded: RMSE $2,776 | R² 0.929


## 2.6 Final Results

Did your model improve?

Yes, the expanded model improved the predictions. The RMSE decreased from $3,408 in the baseline model to $2,776 in the expanded model, meaning the model's prediction errors became smaller. The R² also increased from 0.894 to 0.929, showing that the expanded model explained more of the variation in vehicle selling prices. Adding information from the listing descriptions helped improve the model by including additional details about ownership, accident history, recent service, and premium packages that were not captured by the original variables.